In [60]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

In [88]:
@dataclass
class mlpConfig:
    p: int
    n: int

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.l1 = nn.Linear(in_features=config.p, out_features=1)
    
    def forward(self, x, targets = None):
        # x = self.l1(x)
        
        if targets is not None:
            preds = self.l1(x)
            loss = F.mse_loss(preds, targets)
        else:
            preds = self.l1(x)
            loss = None
        
        return preds, loss

In [89]:
m = nn.Linear(20, 1)
input = torch.randn(128, 20)
output = m(input)
print(output.size())

torch.Size([128, 1])


In [78]:
print(input[0])
print(output[0])

tensor([-0.5178, -2.4554,  0.5339, -1.2895, -2.1008,  0.6718,  1.0156,  0.0973,
         0.0685,  1.0003, -0.6044,  1.5302,  0.4601, -0.0661,  1.2129, -0.0035,
        -0.7426, -1.4485, -1.1337, -0.8849])
tensor([-0.2869], grad_fn=<SelectBackward0>)


In [79]:
import pandas as pd

In [80]:
data = pd.read_csv("/Users/samswitz/GitHub/rthon/test_data/prostate.csv")
info = pd.read_csv("/Users/samswitz/GitHub/rthon/test_data/tidy_coefficients.csv")

In [81]:
data.head()

,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa
0,-0.579819,2.7695,50,-1.386294,0,-1.38629,6,0,-0.43078
1,-0.994252,3.3196,58,-1.386294,0,-1.38629,6,0,-0.16252
2,-0.510826,2.6912,74,-1.386294,0,-1.38629,7,20,-0.16252
3,-1.203973,3.2828,58,-1.386294,0,-1.38629,6,0,-0.16252
4,0.751416,3.4324,62,-1.386294,0,-1.38629,6,0,0.37156


In [67]:
data.iloc[:,-1].values

array([-0.43078, -0.16252, -0.16252, -0.16252,  0.37156,  0.76547,
        0.76547,  0.85442,  1.04732,  1.04732,  1.26695,  1.26695,
        1.26695,  1.34807,  1.39872,  1.44692,  1.47018,  1.4929 ,
        1.55814,  1.59939,  1.639  ,  1.65823,  1.69562,  1.7138 ,
        1.73166,  1.76644,  1.80006,  1.81645,  1.84845,  1.89462,
        1.92425,  2.00821,  2.00821,  2.02155,  2.04769,  2.08567,
        2.15756,  2.19165,  2.21375,  2.27727,  2.29757,  2.30757,
        2.32728,  2.37491,  2.52172,  2.55334,  2.56879,  2.56879,
        2.59152,  2.59152,  2.65676,  2.67759,  2.68444,  2.69124,
        2.70471,  2.718  ,  2.78809,  2.79423,  2.80639,  2.81241,
        2.842  ,  2.85359,  2.85359,  2.882  ,  2.882  ,  2.88759,
        2.92047,  2.96269,  2.96269,  2.97298,  3.01308,  3.03735,
        3.05636,  3.07501,  3.27526,  3.33755,  3.39283,  3.4356 ,
        3.45789,  3.51304,  3.51601,  3.53076,  3.5653 ,  3.57094,
        3.58768,  3.63099,  3.68009,  3.71235,  3.98434,  3.99

In [ ]:
X = torch.tensor(data.iloc[:,:8].values, dtype=torch.float)
X = torch.cat([torch.ones(X.shape[0], 1), X], dim=1)
y = torch.tensor(data.iloc[:,-1].values, dtype=torch.float).view(y.shape[0], -1)

config = mlpConfig(p=X.shape[1], n=X.shape[0])
model = MLP(config)

# with torch.no_grad():
#     model.weight.zero_()
#     model.bias.zero_()

AttributeError: 'MLP' object has no attribute 'bias'

In [106]:
learning_rate = 3e-4
max_iters = 20

print(sum(p.numel() for p in model.parameters()), 'parameters')
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=500, line_search_fn="strong_wolfe")

10 parameters


In [ ]:
for iter in range(max_iters):
    optimizer.zero_grad()
    yhat, loss = model(X, y)
    loss.backward()
    optimizer.step()

In [109]:
yhat = model(X)[0]

sum((y-yhat)**2)

tensor([5311.9321], grad_fn=<AddBackward0>)

In [110]:
for param in model.parameters():
    print(param.data)

tensor([[ 0.2371,  0.0511,  0.2453,  0.0750,  0.1374,  0.3107,  0.3210, -0.3358,
          0.1716]])
tensor([0.1771])
